In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [19]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [20]:
train_dataset = (torchvision.datasets.CIFAR10(root='/data', train=True, download=True, transform=transform))
test_dataset = (torchvision.datasets.CIFAR10(root='/data', train=False, download=True, transform=transform))

In [21]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=64)

CNN Build

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),     # Kernal size =2, stride =2
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.Fucntion_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)    # Flattening
        x = self.Fucntion_layers(x)

        return x

In [26]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimzer = optim.Adam(model.parameters())

In [28]:
# Training Model

epochs = 10

for epoch in range(epochs):
    training_loss = 0.0
    for xb, yb, in train_loader:
        optimzer.zero_grad()

        output = model(xb) # Forward Propagation
        loss   = criterion(output, yb)  # Loss fnx
        loss.backward() # BackPropagation
        optimzer.step() # Update Params

        training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss ={training_loss/len(train_loader)}")

epoch=1/10 & loss =0.11756463839655833
epoch=2/10 & loss =0.10307507671575393
epoch=3/10 & loss =0.08891258654577652
epoch=4/10 & loss =0.08579152426920603
epoch=5/10 & loss =0.08030943075061568
epoch=6/10 & loss =0.06889602825697035
epoch=7/10 & loss =0.07255469684181211
epoch=8/10 & loss =0.06586315375312096
epoch=9/10 & loss =0.06997694970344377
epoch=10/10 & loss =0.05986292114096653


In [29]:
# Evaluate our CNN

correct_labels = 0
total_labels   = 0

model.eval()

with torch.no_grad():
    for xb, yb in test_loader:
        output = model.forward(xb)
        _, predicted = torch.max(output, 1)

        correct_labels += (predicted == yb).sum().item()
        total_labels += yb.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")

Accuracy = 75.31
